# YOLO11m 최종 모델 학습·평가 노트북 (발표용)
**Run All 한 번**으로 아래가 전부 출력됩니다.
1. Roboflow 데이터셋 다운로드 (YOLO11 형식)
2. YOLO11m 학습 (epochs=10)
3. 검증 성능 (mAP@50 / mAP@50-95 / mAP@75 + 클래스별 표)
4. 추론 속도 벤치마크 (FPS, ms/image) — Faster R-CNN / RT-DETR 노트북과 동일한 방식
5. 탐지 결과 이미지 (검증셋 예측 시각화)
6. 학습 곡선 · Confusion Matrix · PR Curve
7. 발표용 최종 요약

> ⚠️ **2번 셀**의 Roboflow 코드만 본인 것(YOLOv11 형식으로 복사한 스니펫)으로 바꾸면 됩니다. 이미 프로젝트 스니펫이 채워져 있어서 그대로 돌려도 됩니다.

In [ ]:
# ============================================================
# 1. 설치
# ============================================================
!pip install -q roboflow ultralytics

In [ ]:
# ============================================================
# 2. Roboflow 데이터셋 다운로드 (YOLO11 형식)
#    ▼▼▼ 필요하면 이 부분만 본인 Roboflow 스니펫으로 교체 ▼▼▼
# ============================================================
from roboflow import Roboflow

rf = Roboflow(api_key="byjMEhXUKbKas3eLE2V9")
project = rf.workspace("elecourse-atjrg").project("merge_0626_3")
version = project.version(13)
dataset = version.download("yolov11")

DATASET_ROOT = dataset.location
print("DATASET_ROOT:", DATASET_ROOT)

In [ ]:
# ============================================================
# 3. data.yaml 경로 보정 (절대경로 yaml 새로 생성)
# ============================================================
import os, glob, yaml
from pathlib import Path

yaml_candidates = glob.glob(os.path.join(DATASET_ROOT, "data.yaml"))
if len(yaml_candidates) == 0:
    yaml_candidates = glob.glob(os.path.join(DATASET_ROOT, "**", "data.yaml"), recursive=True)
if len(yaml_candidates) == 0:
    raise FileNotFoundError("data.yaml을 찾을 수 없습니다.")

DATA_YAML = yaml_candidates[0]
with open(DATA_YAML, "r") as f:
    data = yaml.safe_load(f)

root = Path(DATASET_ROOT)
fixed = data.copy()
fixed["path"] = str(root)
fixed["train"] = "train/images"

if (root / "valid" / "images").exists():
    fixed["val"] = "valid/images"
elif (root / "val" / "images").exists():
    fixed["val"] = "val/images"
elif (root / "test" / "images").exists():
    fixed["val"] = "test/images"
else:
    raise FileNotFoundError("valid/val/test images 폴더를 찾을 수 없습니다.")

if (root / "test" / "images").exists():
    fixed["test"] = "test/images"

FIXED_DATA_YAML = str(root / "yolo11m_data_fixed.yaml")
with open(FIXED_DATA_YAML, "w") as f:
    yaml.safe_dump(fixed, f, sort_keys=False, allow_unicode=True)

print("FIXED_DATA_YAML:", FIXED_DATA_YAML)
print(fixed)

In [ ]:
# ============================================================
# 4. YOLO11m 학습 (epochs=10)
# ============================================================
import torch
from ultralytics import YOLO

print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

model = YOLO("yolo11m.pt")

results = model.train(
    data=FIXED_DATA_YAML,
    epochs=10,
    imgsz=640,
    batch=8,                    # GPU 메모리에 맞게 조정 (OOM 나면 4로)
    device=0 if torch.cuda.is_available() else "cpu",
    project="yolo11m_runs",
    name="yolo11m_final",
    exist_ok=True,
    patience=10,
    plots=True,
)

SAVE_DIR = str(results.save_dir)
BEST_PT = os.path.join(SAVE_DIR, "weights", "best.pt")
print("\nSAVE_DIR:", SAVE_DIR)
print("BEST_PT:", BEST_PT)

In [ ]:
# ============================================================
# 5. 검증 성능 평가 (mAP + 클래스별 표)
# ============================================================
best_model = YOLO(BEST_PT)
metrics = best_model.val(data=FIXED_DATA_YAML, imgsz=640, split="val")

mAP50    = metrics.box.map50
mAP5095  = metrics.box.map
mAP75    = metrics.box.map75
precision = metrics.results_dict["metrics/precision(B)"]
recall    = metrics.results_dict["metrics/recall(B)"]

print("\n========== 전체 검증 성능 ==========")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"mAP@50    : {mAP50:.4f}")
print(f"mAP@50-95 : {mAP5095:.4f}")
print(f"mAP@75    : {mAP75:.4f}")

print("\n========== 클래스별 mAP@50-95 ==========")
names = metrics.names
for cls_idx in metrics.box.ap_class_index:
    print(f"{names[int(cls_idx)]:<20s}  mAP50-95: {metrics.box.maps[int(cls_idx)]:.3f}")

In [ ]:
# ============================================================
# 6. 추론 속도 벤치마크 (FPS / ms per image)
#    - Faster R-CNN, RT-DETR 노트북과 동일하게 검증 이미지 50장 기준
# ============================================================
import time, glob as g

val_img_dir = os.path.join(DATASET_ROOT, fixed["val"])
val_images = sorted(g.glob(os.path.join(val_img_dir, "*.jpg")) + g.glob(os.path.join(val_img_dir, "*.png")))
bench_images = val_images[:50]
print("측정 이미지 수:", len(bench_images))

# 워밍업
for p in bench_images[:5]:
    best_model.predict(p, imgsz=640, verbose=False)

t0 = time.time()
for p in bench_images:
    best_model.predict(p, imgsz=640, verbose=False)
total = time.time() - t0

ms_per_img = total / len(bench_images) * 1000
fps = len(bench_images) / total

print(f"총 시간: {total:.3f}s")
print(f"평균 ms/image: {ms_per_img:.2f} ms")
print(f"FPS: {fps:.2f}")

In [ ]:
# ============================================================
# 7. 탐지 결과 이미지 (검증셋 예측 시각화 → 발표 자료용)
# ============================================================
import matplotlib.pyplot as plt
import random
from PIL import Image

random.seed(0)
sample_imgs = random.sample(val_images, min(6, len(val_images)))

pred_results = best_model.predict(sample_imgs, imgsz=640, conf=0.4, save=True,
                                  project="yolo11m_runs", name="predict_samples", exist_ok=True)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, r in zip(axes.flat, pred_results):
    ax.imshow(r.plot()[:, :, ::-1])   # BGR -> RGB
    ax.set_title(os.path.basename(r.path)[:35], fontsize=8)
    ax.axis("off")
plt.suptitle("YOLO11m 탐지 결과 예시 (valid set, conf>0.4)", fontsize=14)
plt.tight_layout()
plt.show()

print("저장 위치:", pred_results[0].save_dir)

In [ ]:
# ============================================================
# 8. 학습 곡선 / Confusion Matrix / PR Curve
# ============================================================
def show_if_exists(path, title):
    if os.path.exists(path):
        plt.figure(figsize=(12, 8))
        plt.imshow(Image.open(path))
        plt.title(title)
        plt.axis("off")
        plt.show()
    else:
        print("없음:", path)

show_if_exists(os.path.join(SAVE_DIR, "results.png"), "학습 곡선 (loss / mAP)")
show_if_exists(os.path.join(SAVE_DIR, "confusion_matrix_normalized.png"), "Confusion Matrix (normalized)")
show_if_exists(os.path.join(SAVE_DIR, "BoxPR_curve.png"), "Precision-Recall Curve")

In [ ]:
# ============================================================
# 9. 발표용 최종 요약
# ============================================================
n_params = sum(p.numel() for p in best_model.model.parameters())

print("========== YOLO11m 발표용 요약 ==========")
print("Model: YOLO11m")
print("Dataset: Roboflow merge_0626_3 version 13 / YOLO11 export")
print(f"Epochs:     10")
print(f"mAP 50-95: {mAP5095:.4f}")
print(f"mAP 50:    {mAP50:.4f}")
print(f"mAP 75:    {mAP75:.4f}")
print(f"FPS:        {fps:.2f}")
print(f"ms/image:   {ms_per_img:.2f}")
print(f"Params:     {n_params:,}")
print(f"best.pt: {BEST_PT}")
print("==========================================")